# 🎙 Stimmenkloner – Sprachboard (XTTS v2)

Besser als F5-TTS: XTTS v2 wurde speziell für Deutsch und viele andere Sprachen trainiert.

## ⚡ Vor dem Start
**Laufzeit → Laufzeittyp ändern → T4 GPU → Speichern**

## ⚠️ Wichtig: Dieses Notebook hat einen Neustart in der Mitte
- **Phase 1** (Schritte 1–2): Sprachprobe hochladen + Python-Version vorbereiten
- **Kernel startet automatisch neu** nach Schritt 2
- **Phase 2** (Schritte 3–5): Dateinamen eintragen + generieren + herunterladen

---
# PHASE 1 – Vor dem Neustart

## Schritt 1 – Sprachprobe hochladen

- Mindestens **10–30 Sekunden** reine Sprache
- Format: MP3 oder WAV, kein Hintergrundgeräusch

**➜ Nach dem Upload den genauen Dateinamen merken (wird nach Neustart gebraucht)**

In [ ]:
from google.colab import files

print('Datei auswählen...')
uploaded = files.upload()
voice_file = list(uploaded.keys())[0]

print(f'\n✓ Hochgeladen: {voice_file}')
print(f'\n📋 DIESEN NAMEN MERKEN: "{voice_file}"')
print('   (wird in Schritt 3 nach dem Neustart benötigt)')

---
## Schritt 2 – Python 3.10 aktivieren

XTTS v2 benötigt Python 3.10. Dieser Schritt richtet das ein.

**⚠️ Der Kernel startet danach automatisch neu – das ist normal und gewollt.**

Die hochgeladene Datei bleibt erhalten. Du musst nur in Schritt 3 den Dateinamen eintragen.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()  # ← Kernel startet jetzt neu. Danach bei Schritt 3 weitermachen.

---
# PHASE 2 – Nach dem Neustart hier weitermachen

## Schritt 3 – Dateinamen eintragen + Installation

**➜ Den Dateinamen aus Schritt 1 in die erste Zeile eintragen** (z.B. `alex_stimme.mp3`)

In [ ]:
# ↓↓↓ HIER den Dateinamen aus Schritt 1 eintragen ↓↓↓
voice_file = 'DEIN_DATEINAME.mp3'
# ↑↑↑ z.B. 'aufnahme.mp3' oder 'stimme.wav' ↑↑↑

import os
if not os.path.exists(voice_file):
    print(f'❌ Datei "{voice_file}" nicht gefunden!')
    print('   Prüfe ob der Dateiname korrekt ist.')
else:
    print(f'✓ Datei gefunden: {voice_file}')
    print('\nInstalliere XTTS v2...')
    os.system('pip install -q TTS')
    os.system('apt-get install -q ffmpeg')
    print('✓ Installation abgeschlossen')

---
## Schritt 4 – Alle 8 Sätze generieren

Beim ersten Mal wird das XTTS v2 Modell heruntergeladen (~2 GB, ca. 3 Minuten).

Die Sätze kannst du hier beliebig ändern.

In [ ]:
import os
import torch
from TTS.api import TTS

saetze = [
    'Wer auf Toilette möchte, hebt bitte die Hand.',
    'Heute geht die erste Runde Bier selbstverständlich auf mich.',
    'Der Herr ist mein Hirte. Mein Fahrer ist heute Pascal.',
    'Ich erkenne ein gutes Auto daran, wie bequem der Beifahrersitz ist.',
    'Mein Lieblingsauto ist das, in dem mich andere mitnehmen.',
    'Alkoholische Mitarbeit ist heute ausdrücklich erwünscht.',
    'Ich fühle mich wie 2012 im Bierkönig.',
    'Mein Verantwortungsbereich endet ab dem zweiten Bier.',
]

os.environ['COQUI_TOS_AGREED'] = '1'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Verwende: {device.upper()}')

print('Lade XTTS v2 Modell...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('✓ Modell bereit\n')

print(f'Generiere {len(saetze)} Sätze auf Deutsch...\n')
for i, text in enumerate(saetze, 1):
    wav_path = f'/content/clip_{i:02d}.wav'
    mp3_path = f'/content/clip_{i:02d}.mp3'

    tts.tts_to_file(
        text=text,
        speaker_wav=voice_file,
        language='de',
        file_path=wav_path
    )

    os.system(f'ffmpeg -i {wav_path} -q:a 2 {mp3_path} -y -loglevel quiet')
    os.remove(wav_path)

    preview = text[:60] + '...' if len(text) > 60 else text
    print(f'  ✓ clip_{i:02d}.mp3 → {preview}')

print('\n✓ Alle Clips fertig!')

---
## Schritt 5 – Herunterladen

In [ ]:
!zip -j /content/sprachboard_clips.zip /content/clip_*.mp3

from google.colab import files
files.download('/content/sprachboard_clips.zip')
print('✓ Download gestartet: sprachboard_clips.zip')

---
## Nächste Schritte
1. ZIP entpacken → `clip_01.mp3` bis `clip_08.mp3`
2. Alte Clips auf GitHub in `audio/` ersetzen
3. Website spielt automatisch die neuen Clips